In [3]:
import zipfile

zip_path = "spider_data.zip"
extract_to = "."

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_to)

print("Extraction complete.")

Extraction complete.


In [4]:
!pip install -q pandas numpy matplotlib scikit-learn datasets transformers accelerate sqlparse


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [13]:
import os
import re
import json
import math
import random
from pathlib import Path
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset, DatasetDict, load_dataset

import torch
from transformers import AutoTokenizer, AutoConfig

In [28]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MAX_EXAMPLES_TO_PRINT = 3

In [15]:
def get_gpu_report():
    report = {
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count(),
    }

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            report[f"gpu_{i}"] = {
                "name": props.name,
                "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),
                "major": props.major,
                "minor": props.minor,
                "multi_processor_count": props.multi_processor_count,
            }

        report["bf16_supported"] = torch.cuda.is_bf16_supported()
        report["current_device"] = torch.cuda.current_device()

    return report


gpu_report = get_gpu_report()
gpu_report

{'cuda_available': True,
 'device_count': 1,
 'gpu_0': {'name': 'NVIDIA RTX 5000 Ada Generation',
  'total_memory_gb': 31.48,
  'major': 8,
  'minor': 9,
  'multi_processor_count': 100},
 'bf16_supported': True,
 'current_device': 0}

In [17]:
def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_spider_local(spider_dir: Path):

    spider_dir = Path(spider_dir)
    train_path = spider_dir / "train_spider.json"
    train_others_path = spider_dir / "train_others.json"
    dev_path = spider_dir / "dev.json"
    tables_path = spider_dir / "tables.json"

    assert train_path.exists(), f"Missing: {train_path}"
    assert dev_path.exists(), f"Missing: {dev_path}"
    assert tables_path.exists(), f"Missing: {tables_path}"

    train_data = read_json(train_path)
    dev_data = read_json(dev_path)
    tables = read_json(tables_path)

    # Optional additional train file
    if train_others_path.exists():
        train_others = read_json(train_others_path)
        print(f"Found train_others.json with {len(train_others)} examples.")
    else:
        train_others = []
        print("train_others.json not found. Using train_spider.json only.")

    train_df = pd.DataFrame(train_data)
    train_others_df = pd.DataFrame(train_others) if train_others else pd.DataFrame()
    dev_df = pd.DataFrame(dev_data)
    tables_df = pd.DataFrame(tables)

    train_df["split_source"] = "train_spider"
    if len(train_others_df) > 0:
        train_others_df["split_source"] = "train_others"

    dev_df["split_source"] = "dev"

    if len(train_others_df) > 0:
        train_full_df = pd.concat([train_df, train_others_df], ignore_index=True)
    else:
        train_full_df = train_df.copy()

    return train_full_df, dev_df, tables_df


train_df, dev_df, tables_df = load_spider_local("spider_data")

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Tables:", tables_df.shape)

Found train_others.json with 1659 examples.
Train: (8659, 8)
Dev: (1034, 8)
Tables: (166, 8)


In [18]:
train_df.head()

,db_id,query,query_toks,query_toks_no_value,question,question_toks,sql,split_source
0,department_management,SELECT count(*) FROM head WHERE age > 56,"[SELECT, count, (, *, ), FROM, head, WHERE, ag...","[select, count, (, *, ), from, head, where, ag...",How many heads of the departments are older th...,"[How, many, heads, of, the, departments, are, ...","{'from': {'table_units': [['table_unit', 1]], ...",train_spider
1,department_management,"SELECT name , born_state , age FROM head ORD...","[SELECT, name, ,, born_state, ,, age, FROM, he...","[select, name, ,, born_state, ,, age, from, he...","List the name, born state and age of the heads...","[List, the, name, ,, born, state, and, age, of...","{'from': {'table_units': [['table_unit', 1]], ...",train_spider
2,department_management,"SELECT creation , name , budget_in_billions ...","[SELECT, creation, ,, name, ,, budget_in_billi...","[select, creation, ,, name, ,, budget_in_billi...","List the creation year, name and budget of eac...","[List, the, creation, year, ,, name, and, budg...","{'from': {'table_units': [['table_unit', 0]], ...",train_spider
3,department_management,"SELECT max(budget_in_billions) , min(budget_i...","[SELECT, max, (, budget_in_billions, ), ,, min...","[select, max, (, budget_in_billions, ), ,, min...",What are the maximum and minimum budget of the...,"[What, are, the, maximum, and, minimum, budget...","{'from': {'table_units': [['table_unit', 0]], ...",train_spider
4,department_management,SELECT avg(num_employees) FROM department WHER...,"[SELECT, avg, (, num_employees, ), FROM, depar...","[select, avg, (, num_employees, ), from, depar...",What is the average number of employees of the...,"[What, is, the, average, number, of, employees...","{'from': {'table_units': [['table_unit', 0]], ...",train_spider


In [19]:
dev_df.head()

,db_id,query,query_toks,query_toks_no_value,question,question_toks,sql,split_source
0,concert_singer,SELECT count(*) FROM singer,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]",How many singers do we have?,"[How, many, singers, do, we, have, ?]","{'from': {'table_units': [['table_unit', 1]], ...",dev
1,concert_singer,SELECT count(*) FROM singer,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]",What is the total number of singers?,"[What, is, the, total, number, of, singers, ?]","{'from': {'table_units': [['table_unit', 1]], ...",dev
2,concert_singer,"SELECT name , country , age FROM singer ORDE...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","Show name, country, age for all singers ordere...","[Show, name, ,, country, ,, age, for, all, sin...","{'from': {'table_units': [['table_unit', 1]], ...",dev
3,concert_singer,"SELECT name , country , age FROM singer ORDE...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","What are the names, countries, and ages for ev...","[What, are, the, names, ,, countries, ,, and, ...","{'from': {'table_units': [['table_unit', 1]], ...",dev
4,concert_singer,"SELECT avg(age) , min(age) , max(age) FROM s...","[SELECT, avg, (, age, ), ,, min, (, age, ), ,,...","[select, avg, (, age, ), ,, min, (, age, ), ,,...","What is the average, minimum, and maximum age ...","[What, is, the, average, ,, minimum, ,, and, m...","{'from': {'table_units': [['table_unit', 1]], ...",dev


In [20]:
tables_df.head()

,column_names,column_names_original,column_types,db_id,foreign_keys,primary_keys,table_names,table_names_original
0,"[[-1, *], [0, perpetrator id], [0, people id],...","[[-1, *], [0, Perpetrator_ID], [0, People_ID],...","[text, number, number, text, number, text, tex...",perpetrator,"[[2, 9]]","[1, 9]","[perpetrator, people]","[perpetrator, people]"
1,"[[-1, *], [0, building], [0, room number], [0,...","[[-1, *], [0, building], [0, room_number], [0,...","[text, text, text, number, text, text, number,...",college_2,"[[9, 4], [13, 4], [19, 1], [20, 2], [15, 7], [...","[1, 4, 7, 11, 15, 22, 27, 31, 37, 39, 45]","[classroom, department, course, instructor, se...","[classroom, department, course, instructor, se..."
2,"[[-1, *], [0, id], [0, city], [0, country], [0...","[[-1, *], [0, id], [0, City], [0, Country], [0...","[text, number, text, text, text, text, text, n...",flight_company,"[[20, 7], [19, 1]]","[1, 7, 13]","[airport, operate company, flight]","[airport, operate_company, flight]"
3,"[[-1, *], [0, institution id], [0, name], [0, ...","[[-1, *], [0, instID], [0, name], [0, country]...","[text, number, text, text, number, text, text,...",icfp_1,"[[11, 7], [10, 1], [9, 4]]","[1, 4, 7, 9]","[institution, authors, papers, authorship count]","[Inst, Authors, Papers, Authorship]"
4,"[[-1, *], [0, body builder id], [0, people id]...","[[-1, *], [0, Body_Builder_ID], [0, People_ID]...","[text, number, number, number, number, number,...",body_builder,"[[2, 6]]","[1, 6]","[body builder, people]","[body_builder, people]"


In [22]:
def basic_quality_report(df, name="dataset"):
    print(f"===== {name} =====")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nNull counts:")
    print(df.isna().sum())

    print("\nUnique DBs:", df["db_id"].nunique())
    print("Duplicate question+query rows:", df.duplicated(subset=["db_id", "question", "query"]).sum())

    print("\nSample rows:")
    display(df[["db_id", "question", "query"]].head(3))


basic_quality_report(train_df, "Train")
basic_quality_report(dev_df, "Dev")

===== Train =====
Shape: (8659, 8)

Columns:
['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql', 'split_source']

Null counts:
db_id                  0
query                  0
query_toks             0
query_toks_no_value    0
question               0
question_toks          0
sql                    0
split_source           0
dtype: int64

Unique DBs: 146
Duplicate question+query rows: 8

Sample rows:


,db_id,question,query
0,department_management,How many heads of the departments are older th...,SELECT count(*) FROM head WHERE age > 56
1,department_management,"List the name, born state and age of the heads...","SELECT name , born_state , age FROM head ORD..."
2,department_management,"List the creation year, name and budget of eac...","SELECT creation , name , budget_in_billions ..."


===== Dev =====
Shape: (1034, 8)

Columns:
['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql', 'split_source']

Null counts:
db_id                  0
query                  0
query_toks             0
query_toks_no_value    0
question               0
question_toks          0
sql                    0
split_source           0
dtype: int64

Unique DBs: 20
Duplicate question+query rows: 0

Sample rows:


,db_id,question,query
0,concert_singer,How many singers do we have?,SELECT count(*) FROM singer
1,concert_singer,What is the total number of singers?,SELECT count(*) FROM singer
2,concert_singer,"Show name, country, age for all singers ordere...","SELECT name , country , age FROM singer ORDE..."


In [31]:
def build_schema_map(tables_df: pd.DataFrame):
    """
    Converts Spider tables.json into readable schema strings by db_id.
    Produces compact structured schema text suitable for humans + LLM prompts.
    """

    schema_map = {}

    for _, row in tables_df.iterrows():

        db_id = row["db_id"]

        table_names = row["table_names_original"]
        column_names = row["column_names_original"]

        column_types = row.get("column_types", [])

        primary_keys = set(row.get("primary_keys", []))
        foreign_keys = row.get("foreign_keys", [])

        # Build FK lookup
        fk_lookup = {}

        for fk_col_idx, ref_col_idx in foreign_keys:

            try:
                fk_table_idx, fk_col_name = column_names[fk_col_idx]
                ref_table_idx, ref_col_name = column_names[ref_col_idx]

                if fk_table_idx != -1 and ref_table_idx != -1:

                    ref_table = table_names[ref_table_idx]

                    fk_lookup[fk_col_idx] = (
                        f" [FK -> {ref_table}.{ref_col_name}]"
                    )

            except Exception:
                pass

        table_to_columns = defaultdict(list)

        # Build table columns
        for col_idx, (table_idx, col_name) in enumerate(column_names):

            # Spider uses -1 for "*"
            if table_idx == -1:
                continue

            table_name = table_names[table_idx]

            col_type = (
                column_types[col_idx]
                if col_idx < len(column_types)
                else "text"
            )

            pk_marker = " [PK]" if col_idx in primary_keys else ""

            fk_marker = fk_lookup.get(col_idx, "")

            col_text = (
                f"{col_name}:{col_type}"
                f"{pk_marker}"
                f"{fk_marker}"
            )

            table_to_columns[table_name].append(col_text)

        # Build final schema text
        lines = [
            f"Database: {db_id}",
            "Schema:"
        ]

        for table_name, cols in table_to_columns.items():

            lines.append(f"\nTable {table_name}:")
            
            for col in cols:
                lines.append(f"  - {col}")

        schema_map[db_id] = "\n".join(lines)

    return schema_map


# Build schema map
schema_map = build_schema_map(tables_df)

# Example preview
sample_db = list(schema_map.keys())[0]

print(schema_map[sample_db][:3000])

Database: perpetrator
Schema:

Table perpetrator:
  - Perpetrator_ID:number [PK]
  - People_ID:number [FK -> people.People_ID]
  - Date:text
  - Year:number
  - Location:text
  - Country:text
  - Killed:number
  - Injured:number

Table people:
  - People_ID:number [PK]
  - Name:text
  - Height:number
  - Weight:number
  - Home Town:text


In [32]:
def attach_schema(df, schema_map):
    df = df.copy()
    df["schema"] = df["db_id"].map(schema_map)

    missing_schema = df["schema"].isna().sum()
    if missing_schema > 0:
        print(f"Warning: {missing_schema} rows missing schema.")

    return df


train_df = attach_schema(train_df, schema_map)
dev_df = attach_schema(dev_df, schema_map)

train_df[["db_id", "question", "query", "schema"]].head(2)

,db_id,question,query,schema
0,department_management,How many heads of the departments are older than 56 ?,SELECT count(*) FROM head WHERE age > 56,Database: department_management\nSchema:\n\nTable department:\n - Department_ID:number [PK]\n - Name:text\n - Creation:text\n - Ranking:number\n - Budget_in_Billions:number\n - Num_Employees:number\n\nTable head:\n - head_ID:number [PK]\n - name:text\n - born_state:text\n - age:number\n\nTable management:\n - department_ID:number [PK] [FK -> department.Department_ID]\n - head_ID:number [FK -> head.head_ID]\n - temporary_acting:text
1,department_management,"List the name, born state and age of the heads of departments ordered by age.","SELECT name , born_state , age FROM head ORDER BY age",Database: department_management\nSchema:\n\nTable department:\n - Department_ID:number [PK]\n - Name:text\n - Creation:text\n - Ranking:number\n - Budget_in_Billions:number\n - Num_Employees:number\n\nTable head:\n - head_ID:number [PK]\n - name:text\n - born_state:text\n - age:number\n\nTable management:\n - department_ID:number [PK] [FK -> department.Department_ID]\n - head_ID:number [FK -> head.head_ID]\n - temporary_acting:text


In [33]:
SYSTEM_PROMPT = """You are a text-to-SQL assistant. 
Generate a valid SQL query for the user's question using only the provided database schema.
Return only the SQL query without explanation.
"""


def create_prompt(question, schema):
    return f"""### System:
{SYSTEM_PROMPT.strip()}

### Database Schema:
{schema}

### User Question:
{question}

### SQL:
"""


def create_sft_text(question, schema, sql):
    prompt = create_prompt(question, schema)
    return prompt + sql.strip()


def prepare_sft_dataframe(df):
    df = df.copy()

    df["prompt"] = df.apply(
        lambda x: create_prompt(x["question"], x["schema"]),
        axis=1
    )

    df["completion"] = df["query"].astype(str).str.strip()

    df["text"] = df.apply(
        lambda x: create_sft_text(x["question"], x["schema"], x["query"]),
        axis=1
    )

    return df


train_sft_df = prepare_sft_dataframe(train_df)
dev_sft_df = prepare_sft_dataframe(dev_df)

In [34]:
print(train_sft_df["text"].iloc[0][:2000])

### System:
You are a text-to-SQL assistant. 
Generate a valid SQL query for the user's question using only the provided database schema.
Return only the SQL query without explanation.

### Database Schema:
Database: department_management
Schema:

Table department:
  - Department_ID:number [PK]
  - Name:text
  - Creation:text
  - Ranking:number
  - Budget_in_Billions:number
  - Num_Employees:number

Table head:
  - head_ID:number [PK]
  - name:text
  - born_state:text
  - age:number

Table management:
  - department_ID:number [PK] [FK -> department.Department_ID]
  - head_ID:number [FK -> head.head_ID]
  - temporary_acting:text

### User Question:
How many heads of the departments are older than 56 ?

### SQL:
SELECT count(*) FROM head WHERE age  >  56
